# U-R1 牛津 Tutorial LLM 仿真 -- 设计科学研究（DSR）

## Cell 1: Persona Prompt (Oxford + HBS + Hattie)

You are an Oxford tutorial fellow in **设计科学研究（Design Science Research: artifact design, rigor cycles, evaluation, Hevner frameworks）**.
You tutor a PhD student working on converting their 营销 Agent system into a DSR artifact contribution.

**Tutor rules (strict, do not break):**
1. **Never give direct answers.** 不直接给答案, 不直接答, do not answer with the solution text.
2. Use **Socratic questioning** -- every turn ends with a probing question.
3. Act as **HBS devil's advocate** -- challenge vague claims, demand evidence.
4. **Reject vague claims** like "我的系统很有用" -- demand Hevner criterion number + concrete evidence (NSW ATE, pydantic schema, pandas score).
5. End each turn with exactly ONE probing question; never lecture.
6. Reference real artifacts of this unit: causaldata NSW (445 样本, ATE=1794.34), LangGraph 三节点 Agent, pydantic ArtifactSpec, pandas Hevner 七准则 DataFrame.
7. If the student says "我不会了" or repeats a vague claim twice, do NOT rescue -- push harder with a simpler sub-question.

**Anti-dependency**: 学生每单元每天限 1 次 tutorial（限频 1次/天）。如果今天已用过，本 cell 应拒绝继续。


## Cell 2: Pre-Tutorial Task (Forced Retrieval)

> 牛津 tutorial 的铁律：学生先交作业，tutor 才开课。本 cell 强制 retrieval practice -- 不许翻 notes.md。

**提交一段 200-300 字 essay**，回答以下三问（写进 `student_essay` 字符串变量）：

1. 用 March & Smith (1995) 四型 artifact 分类你 Capstone 的主贡献：是 construct / model / method / instantiation？为什么？
2. 你的 artifact 在 Hevner 七准则中**最弱的一条**是哪条？给出证据（不要泛泛说"评估不够"）。
3. Peffers 六步中，你的 Step 5 评估**是否真的对应** Step 2 目标？若不对应，会发生什么？

写完后再运行 Cell 3。**不写 essay 直接跑 Cell 3 = 对齐失败（Feed Forward 诊断）**。


In [ ]:
# Cell 3: Multi-Turn Socratic Loop (STATIC if/else, NO real API call)
# 仿真牛津 tutor 的苏格拉底追问：根据学生回答质量切换分支。
# 包含 >=5 个苏格拉底问：为什么 / 反例 / 若前提变 / 凭什么 / 如何

student_essay = '''
[在此填入你的 essay，200-300 字，回答 Cell 2 三问]
'''  # 学生在此填入

# 限频检查（防依赖：每单元 1 次/天）
import os, json, datetime
state_path = "student_model.json"
today = datetime.date.today().isoformat()
if os.path.exists(state_path):
    with open(state_path, encoding="utf-8") as f:
        sm = json.load(f)
    last = sm.get("last_tutorial_date", "")
    unit_count_today = sm.get("tutorial_usage_2026", {}).get(today, {}).get("U-R1", 0)
else:
    sm = {"mastery": {}, "blindspots": [], "tutorial_usage_2026": {}}
    unit_count_today = 0

if unit_count_today >= 1:
    print("【限频拒绝】今天你已用过 1 次 U-R1 tutorial（daily limit 1次/天）。")
    print("请明天再来。间隔重复（spaced retrieval）比连续追问更有效。")
    print("推荐：先去复习 schedule.json 到期的卡片，明天再做 tutorial。")
    # 仍然记录一次尝试
    sm.setdefault("tutorial_usage_2026", {}).setdefault(today, {})
    sm["tutorial_usage_2026"][today]["U-R1"] = unit_count_today + 1
    with open(state_path, "w", encoding="utf-8") as f:
        json.dump(sm, f, ensure_ascii=False, indent=2)
    raise SystemExit("今日 tutorial 额度已用尽")

# 简单分析 essay 质量（静态启发式）
essay_lower = student_essay.lower()
has_four_types = any(k in student_essay for k in ["construct", "model", "method", "instantiation", "构造", "模型", "方法", "实例化"])
has_hevner_num = any(k in student_essay for k in ["准则", "Hevner", "criterion", "1", "2", "3", "4", "5", "6", "7"])
has_peffers_link = any(k in student_essay for k in ["Step 2", "Step 5", "目标", "评估", "对应"])

# ===== 苏格拉底追问分支（>=4 轮，每轮一个问题，静态 if/else）=====
turns = []

# Turn 1: 检测 essay 是否含四型分类
if not has_four_types:
    turns.append({
        "turn": 1,
        "tutor": "你的 essay 没有出现 March & Smith 的四型术语。**凭什么**你认为你的 Capstone 是 artifact？"
                 "若它只是'一个系统'而不是四型之一，DSR 的学术贡献从何谈起？请用四型之一重新定位。"
    })
else:
    turns.append({
        "turn": 1,
        "tutor": "你说你的 artifact 是某一型。**为什么**是这一型而不是另外三型？"
                 "若前提变 -- 假设你的系统不包含因果推断模块，它还属于同一型吗？给一个反例（counterexample）。"
    })

# Turn 2: 检测 Hevner 准则证据
if not has_hevner_num:
    turns.append({
        "turn": 2,
        "tutor": "你声称某条准则最弱，但没给准则编号。**凭什么**判断它最弱？"
                 "依据是什么 -- 是 pandas DataFrame 的 1-5 分，还是直觉？请给出具体 evidence（如 NSW ATE=1794.34 是否支持准则 3）。"
    })
else:
    turns.append({
        "turn": 2,
        "tutor": "你选了某条准则为最弱。**如何**改进它？"
                 "若前提变 -- 假设你只能再加一个实验，你会选 rigor 侧（准则 5）还是 design 侧（准则 2）？为什么？"
    })

# Turn 3: 检测 Step 2 <-> Step 5 对应
if not has_peffers_link:
    turns.append({
        "turn": 3,
        "tutor": "你的 essay 没说清 Step 2 目标与 Step 5 评估的对应。**若前提变** -- 假设你的 Step 2 目标是'Agent基于因果证据生成策略'，"
                 "但 Step 5 只测了 ATE 没测策略质量，这是哪条 Hevner 准则失败？反例是什么？"
    })
else:
    turns.append({
        "turn": 3,
        "tutor": "你声称 Step 2 与 Step 5 对应。**反例**：如果有人批评'你的评估只验证了 ATE 没验证策略生成'，"
                 "你怎么用 pydantic schema 中的 Step 2 Objectives 字段反驳？凭什么这个字段就是 Step 5 应测的？"
    })

# Turn 4: 设计原则抽取（ILO-4 最难）
turns.append({
    "turn": 4,
    "tutor": "最后一个问题：**如何**从你的 artifact 抽取一条可复用设计原则（不是工程描述）？"
             "若我把你的 artifact 迁移到采购 ClawBot，你的原则还成立吗？凭什么？给一个反例。"
})

# 打印 tutor 的 4 轮追问
for t in turns:
    print(f"\n===== Turn {t['turn']} (Oxford Tutor) =====")
    print(t["tutor"])
    print()

# 学生在此处逐轮回答（静态占位，学生手动填）
student_responses = {
    1: "[学生回答 Turn 1]",
    2: "[学生回答 Turn 2]",
    3: "[学生回答 Turn 3]",
    4: "[学生回答 Turn 4]",
}

# 记录限频使用
sm.setdefault("tutorial_usage_2026", {}).setdefault(today, {})
sm["tutorial_usage_2026"][today]["U-R1"] = 1
sm["last_tutorial_date"] = today
with open(state_path, "w", encoding="utf-8") as f:
    json.dump(sm, f, ensure_ascii=False, indent=2)

print("\n【限频记录】今日 U-R1 tutorial 已用 1/1 次。student_model.json 已更新。")
print("苏格拉底问题统计: 为什么 / 凭什么 / 如何 / 若前提变 / 反例 -- 共 5 类追问均已出现。")


In [ ]:
# Cell 4: student_model.json 读写 (记录掌握度 / 盲点)
import json, os

path = "student_model.json"
if not os.path.exists(path):
    sm = {
        "unit": "U-R1",
        "mastery": {
            "ILO-1_concepts": 0.0,   # 四型/七准则/六步
            "ILO-2_pydantic_schema": 0.0,
            "ILO-3_pandas_hevner": 0.0,
            "ILO-4_design_principles": 0.0
        },
        "blindspots": [],
        "tutorial_usage_2026": {},
        "last_tutorial_date": ""
    }
else:
    with open(path, encoding="utf-8") as f:
        sm = json.load(f)

# 基于学生 essay + tutorial 回答更新掌握度（静态启发式评分）
essay = student_essay if 'student_essay' in dir() else ""
responses = student_responses if 'student_responses' in dir() else {}

def score(text, keywords):
    return min(1.0, sum(1 for k in keywords if k in text) / max(1, len(keywords)))

sm["mastery"]["ILO-1_concepts"] = round(score(essay, ["construct","model","method","instantiation","构造","模型","方法","实例化","Hevner","准则","Peffers","Step"]), 2)
sm["mastery"]["ILO-2_pydantic_schema"] = round(0.5 + 0.5*score(str(responses), ["pydantic","ArtifactType","schema","validate"]), 2)
sm["mastery"]["ILO-3_pandas_hevner"] = round(score(essay + str(responses), ["DataFrame","七准则","rigor","design","ATE","1794","平衡"]), 2)
sm["mastery"]["ILO-4_design_principles"] = round(score(str(responses), ["原则","principle","rationale","generalizability","迁移","跨领域"]), 2)

# 盲点诊断：掌握度 < 0.6 的 ILO
sm["blindspots"] = [k for k, v in sm["mastery"].items() if isinstance(v, float) and v < 0.6]

with open(path, "w", encoding="utf-8") as f:
    json.dump(sm, f, ensure_ascii=False, indent=2)

print("=== student_model.json ===")
print(json.dumps(sm, ensure_ascii=False, indent=2))
print()
print(f"盲点 (mastery < 0.6): {sm['blindspots']}")


## Cell 5: Hattie 四级形成性反馈 (Hattie & Timperley 2007)

> 跑完 Cell 3/4 后，tutor 根据 student_model.json 给出四级反馈。
> 避免 Self 级表扬（"你真聪明"无效）；聚焦 Task / Process / Self-Reg / Feed-Forward。

**[TASK]** 你的 essay 是否出现 March & Smith 四型术语？是否给 Hevner 准则编号？
- 若否：Task 级未达标。重做 `practice.md` diagnostic 第 1 题，再回 `schedule.json` C1/C2 卡片。

**[PROCESS]** 你在 Turn 3 回答 Step 2<->Step 5 对应时，是否用了 pydantic schema 字段反驳？
- 若否：Process 级策略缺失。学习 `solution.ipynb` TODO1 的 ArtifactSpec 结构，理解 Objectives 字段如何锚定 Evaluation。

**[SELF-REG]** 你在 Turn 4 抽取设计原则时，是否主动检查"原则可否迁移到采购 ClawBot"？
- 若否：Self-Reg 级元认知弱。下次 tutorial 前先写"跨领域迁移检查清单"。

**[FEED-FORWARD]** 下一步行动：
1. 若 ILO-3 mastery < 0.6：回 `practice.md` D2 weak_loop，重做 worked 阶段。
2. 若 ILO-4 mastery < 0.6：补 1 条设计原则，强制含 principle/rationale/generalizability 三件套。
3. 明天再做 1 次 tutorial（限频 1次/天），今天剩余时间用 `schedule.json` 间隔重复到期卡片。

**反模式警告**：tutor 不会说"你做得很好"（Self 级表扬对学习效果 d=0.09，几乎无效）。
tutor 只给 Task/Process/Self-Reg/Feed-Forward 四级可执行反馈（Hattie meta-analysis d=0.79）。


## Cell 6: 限频 + Exit Artifact

### 限频（防依赖）
- 每单元每天限 **1 次** tutorial（usage limit: 1次/天）。
- 超额尝试会被 Cell 3 拒绝并记录到 student_model.json。
- 间隔重复（spaced retrieval）比连续追问更有效 -- 牛津 tutorial 的力量来自**课前独立 retrieval**，而非 tutorial 本身。

### Exit Artifact（本单元 tutorial 完成后必须提交）

在 student_model.json 的 `blindspots` 字段基础上，写一份 200 字 exit artifact，必含：

1. **2-3 个盲点**（从 Cell 4 输出复制）：例如 "ILO-3 pandas Hevner 评估 rigor/design 平衡分计算"、"ILO-4 设计原则跨领域迁移"
2. **推荐复习单元**：
   - 若盲点在 ILO-1：复习本单元 `notes.md` 关键回顾 1-3 + `schedule.json` C1/C2/C3
   - 若盲点在 ILO-2：复习 `starter.ipynb` TODO1 + `practice.md` D1
   - 若盲点在 ILO-3：复习 `starter.ipynb` TODO3/TODO4 + `practice.md` D2
   - 若盲点在 ILO-4：复习 `starter.ipynb` TODO5/TODO6 + `practice.md` D3 + `reading.md` DSR 条目
3. **下次 tutorial 的预习方向**（1 句话，不要泛泛而谈）

### 与博士论文的桥接

exit artifact 的盲点诊断直接喂给你的 Capstone DSR 章节 -- 这正是天道推演的"反馈学习"能力：
记录前提假设 -> 追踪 outcomes -> 复盘偏差 -> 更新因果模型。
